# PROJECT BUSINESS UNDERSTANDING  
## Port Trade Disruption Early-Warning System for Kenya

### BUSINESS PROBLEM  
Kenya’s economy depends heavily on port activity, particularly at Mombasa and Lamu. When port operations are disrupted, the effects spread quickly across trade, fuel supply, manufacturing, and food distribution. Current responses are mostly reactive, meaning that stakeholders often act only after losses and delays have already occurred.

### PROBLEM STATEMENT  
This project develops a machine learning early-warning system that predicts significant port trade disruptions using daily port operations data. Its purpose is to provide advance risk signals that can help stakeholders respond earlier, strengthen logistics resilience, and support better decision-making.

### STAKEHOLDERS  
**Kenya Ports Authority:** For operational monitoring and early intervention.  
**Government Agencies:** For trade, customs, and economic response planning.  
**Importers and Exporters:** For shipment and supply chain adjustment.  
**Manufacturers and Retailers:** For production and inventory planning.  
**Consumers:** As indirect beneficiaries of improved supply stability.



| Disruption Type         | What Happens                                      | Example from Data                                                                 | Impact                                      |
|------------------------|--------------------------------------------------|----------------------------------------------------------------------------------------|---------------------------------------------|
| Demand-side disruption | Fewer vessels arrive → cargo volume drops        | COVID-19 (April 2020): Mombasa went from 5–7 port calls/day to 1–2 calls/day           | Importers wait weeks for goods*              |
| Supply-side disruption | Vessels arrive but cannot unload → cargo piles up| Congestion event (Dec 2025): 28 vessels waiting to berth                               | Perishable goods rot, demurrage costs explode |

A disruption occurs when daily cargo throughput (import + export tonnes) falls below 50% of its 30-day rolling average OR when berth occupancy exceeds 90% for 3 consecutive days.

# Section 1: Environment Setup

## 1.1 Import Core Libraries


In [3]:
# Data manipulation
import pandas as pd
import numpy as np

# Date handling
from datetime import datetime, timedelta

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning


# Explainability


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Section 2: Data Understanding and Cleaning

## 2.1 Data loading

In [12]:
df_port = pd.read_csv("port_activity_data.csv")
df_turn = pd.read_csv("SHIP TURN AROUND TIME.csv")
df_wait = pd.read_csv("SHIP WAITING TIME.csv")
print(f"Port daily activities: {df_port.shape[0]} rows {df_port.shape[1]} columns")
print(f"Vessel turn around time: {df_turn.shape[0]} rows {df_turn.shape[1]} columns")
print(f"Vessel waiting time: {df_wait.shape[0]} rows {df_wait.shape[1]} columns")

Port daily activities: 5300 rows 29 columns
Vessel turn around time: 241 rows 5 columns
Vessel waiting time: 245 rows 5 columns


## 2.2 Understanding dataset

### 2.2.1 Port activities dataset

In [18]:
#port activities columns names
print(df_port.columns.T)

Index(['date', 'year', 'month', 'day', 'portid', 'portname', 'country', 'ISO3',
       'portcalls_container', 'portcalls_dry_bulk', 'portcalls_general_cargo',
       'portcalls_roro', 'portcalls_tanker', 'portcalls_cargo', 'portcalls',
       'import_container', 'import_dry_bulk', 'import_general_cargo',
       'import_roro', 'import_tanker', 'import_cargo', 'import',
       'export_container', 'export_dry_bulk', 'export_general_cargo',
       'export_roro', 'export_tanker', 'export_cargo', 'export'],
      dtype='object')


In [17]:
#port activities column data type
print(df_port.dtypes)

date                       object
year                        int64
month                       int64
day                         int64
portid                     object
portname                   object
country                    object
ISO3                       object
portcalls_container         int64
portcalls_dry_bulk          int64
portcalls_general_cargo     int64
portcalls_roro              int64
portcalls_tanker            int64
portcalls_cargo             int64
portcalls                   int64
import_container            int64
import_dry_bulk             int64
import_general_cargo        int64
import_roro                 int64
import_tanker               int64
import_cargo                int64
import                      int64
export_container            int64
export_dry_bulk             int64
export_general_cargo        int64
export_roro                 int64
export_tanker               int64
export_cargo                int64
export                      int64
dtype: object


In [20]:
#port activities sample data
df_port.head()

,date,year,month,day,portid,portname,country,ISO3,portcalls_container,portcalls_dry_bulk,...,import_tanker,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export
0,2026-04-03 00:00:00+00:00,2026,4,3,port631,Lamu,Kenya,KEN,1,0,...,0,15633,15633,0,0,0,0,0,0,0
1,2026-04-03 00:00:00+00:00,2026,4,3,port757,Mombasa,Kenya,KEN,2,0,...,0,24577,24577,1370,0,1910,0,0,3280,3280
2,2026-04-02 00:00:00+00:00,2026,4,2,port631,Lamu,Kenya,KEN,1,0,...,0,0,0,3711,0,0,0,0,3711,3711
3,2026-04-02 00:00:00+00:00,2026,4,2,port757,Mombasa,Kenya,KEN,1,1,...,66065,40876,106941,685,1327,0,0,509,2012,2522
4,2026-04-01 00:00:00+00:00,2026,4,1,port631,Lamu,Kenya,KEN,2,0,...,0,3567,3567,1697,0,0,0,0,1697,1697


In [26]:
df_port["date"].max()

'2026-04-03 00:00:00+00:00'

In [28]:
#date range
print(f"Date ranges from: {df_port['date'].min()} - {df_port['date'].max()}")

Date ranges from: 2019-01-01 00:00:00+00:00 - 2026-04-03 00:00:00+00:00


In [37]:
# missing values
missing = df_port.isnull().sum()
missing[missing >0]

Series([], dtype: int64)

In [ ]:
# duplicate rows
df_port.duplicated().sum()

0

In [38]:
#statistical analysis
df_port.describe()

,year,month,day,portcalls_container,portcalls_dry_bulk,portcalls_general_cargo,portcalls_roro,portcalls_tanker,portcalls_cargo,portcalls,...,import_tanker,import_cargo,import,export_container,export_dry_bulk,export_general_cargo,export_roro,export_tanker,export_cargo,export
count,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,...,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000,5300.000000
mean,2022.140377,6.366038,15.708679,0.795849,0.460189,0.337547,0.180189,0.400000,1.773774,2.173774,...,13855.328302,19797.734151,33653.178868,563.190189,517.922264,239.606038,3.656038,288.545283,1324.403774,1612.975849
std,2.098156,3.489987,8.805699,1.102683,0.721518,0.583333,0.438934,0.707974,1.881855,2.267660,...,31229.992070,30648.221913,49771.260482,2097.754532,4647.114916,809.947318,79.341149,2742.017945,5275.932187,5967.193499
min,2019.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2020.000000,3.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2022.000000,6.000000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,0.000000,56.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2024.000000,9.000000,23.000000,1.000000,1.000000,1.000000,0.000000,1.000000,3.000000,4.000000,...,0.000000,32015.000000,57044.750000,0.000000,0.000000,0.000000,0.000000,0.000000,685.000000,955.000000
max,2026.000000,12.000000,31.000000,7.000000,7.000000,5.000000,4.000000,5.000000,19.000000,24.000000,...,234228.000000,306828.000000,541056.000000,30691.000000,108624.000000,10793.000000,3921.000000,111941.000000,109168.000000,111941.000000
